# Session 3, Module 01: Error Handling


This module covers:
- try / except / else / finally flow
- Catching specific exceptions vs bare except
- Exception hierarchy
- Multiple except blocks
- Raising exceptions with raise

Data Engineering Context:
Error handling is critical for data pipelines. Handle API failures,
database errors, and file not found gracefully.


## Basic Try / Except


In [18]:
print("=== Basic try / except ===")

=== Basic try / except ===


Without error handling - program crashes
result = 10 / 0  # ZeroDivisionError!
With error handling

In [19]:
try:
    result = 10 / 0
except ZeroDivisionError:
    print("Cannot divide by zero!")
    result = None

print(f"Result: {result}")

# Catching the exception object
try:
    result = 10 / 0
except ZeroDivisionError as e:
    print(f"Error type: {type(e).__name__}")
    print(f"Error message: {e}")

Cannot divide by zero!
Result: None
Error type: ZeroDivisionError
Error message: division by zero


## Catching Specific Exceptions


In [3]:
print("\n=== Catching Specific Exceptions ===")


def parse_record(record_str: str) -> dict:
    """Parse a record string into a dictionary."""
    try:
        # Multiple things can go wrong here
        parts = record_str.split(",")
        return {
            "id": int(parts[0]),
            "name": parts[1],
            "value": float(parts[2]),
        }
    except ValueError as e:
        print(f"Value conversion error: {e}")
        return None
    except IndexError as e:
        print(f"Missing fields: {e}")
        return None


# Test with various inputs
test_records = [
    "1,Alice,100.50",      # Valid
    "two,Bob,200",         # Invalid ID
    "3,Charlie",           # Missing field
]

for record_str in test_records:
    print(f"\nParsing: '{record_str}'")
    result = parse_record(record_str)
    print(f"Result: {result}")


=== Catching Specific Exceptions ===

Parsing: '1,Alice,100.50'
Result: {'id': 1, 'name': 'Alice', 'value': 100.5}

Parsing: 'two,Bob,200'
Value conversion error: invalid literal for int() with base 10: 'two'
Result: None

Parsing: '3,Charlie'
Missing fields: list index out of range
Result: None


## Multiple Except Blocks


In [4]:
print("\n=== Multiple Except Blocks ===")


def process_data(data):
    """Process data with multiple error types."""
    try:
        # Various operations that might fail
        if data is None:
            raise ValueError("Data cannot be None")

        if not isinstance(data, dict):
            raise TypeError(f"Expected dict, got {type(data).__name__}")

        value = data["value"]  # Might raise KeyError
        result = 100 / value   # Might raise ZeroDivisionError

        return result

    except ValueError as e:
        print(f"ValueError: {e}")
    except TypeError as e:
        print(f"TypeError: {e}")
    except KeyError as e:
        print(f"KeyError: Missing key {e}")
    except ZeroDivisionError:
        print("ZeroDivisionError: Cannot divide by zero")

    return None


# Test cases
test_cases = [
    {"value": 10},      # Valid
    None,               # ValueError
    "not a dict",       # TypeError
    {"name": "test"},   # KeyError
    {"value": 0},       # ZeroDivisionError
]

for data in test_cases:
    print(f"\nProcessing: {data}")
    result = process_data(data)
    print(f"Result: {result}")


=== Multiple Except Blocks ===

Processing: {'value': 10}
Result: 10.0

Processing: None
ValueError: Data cannot be None
Result: None

Processing: not a dict
TypeError: Expected dict, got str
Result: None

Processing: {'name': 'test'}
KeyError: Missing key 'value'
Result: None

Processing: {'value': 0}
ZeroDivisionError: Cannot divide by zero
Result: None


## Catching Multiple Exceptions In One Block


In [5]:
print("\n=== Catching Multiple Exception Types ===")

try:
    # Some operation
    value = int("not_a_number")
except (ValueError, TypeError) as e:
    # Handle both the same way
    print(f"Conversion error: {e}")


=== Catching Multiple Exception Types ===
Conversion error: invalid literal for int() with base 10: 'not_a_number'


## The Else Clause


In [6]:
print("\n=== The else Clause ===")


=== The else Clause ===


else runs ONLY if no exception occurred
Use it to separate the "try" block from the "success" code

In [7]:
def safe_divide(a: float, b: float) -> float | None:
    """Divide a by b with error handling."""
    try:
        result = a / b
    except ZeroDivisionError:
        print("Cannot divide by zero")
        return None
    else:
        # This runs only if no exception occurred
        print(f"Division successful: {a} / {b} = {result}")
        return result


safe_divide(10, 2)   # Success path
safe_divide(10, 0)   # Error path

Division successful: 10 / 2 = 5.0
Cannot divide by zero


## The Finally Clause


In [8]:
print("\n=== The finally Clause ===")


=== The finally Clause ===


finally ALWAYS runs, whether exception occurred or not
Use for cleanup (closing files, connections, etc.)

In [9]:
def read_file_safely(file_path: str) -> str | None:
    """Read file content with guaranteed cleanup."""
    file_handle = None
    try:
        print(f"Opening {file_path}...")
        file_handle = open(file_path, "r")
        content = file_handle.read()
        return content
    except FileNotFoundError:
        print(f"File not found: {file_path}")
        return None
    except PermissionError:
        print(f"Permission denied: {file_path}")
        return None
    finally:
        # Always runs - cleanup
        if file_handle:
            print("Closing file handle...")
            file_handle.close()
        print("Cleanup complete")


# Test with non-existent file
read_file_safely("/nonexistent/file.txt")

Opening /nonexistent/file.txt...
File not found: /nonexistent/file.txt
Cleanup complete


## Complete Flow: Try / Except / Else / Finally


In [10]:
print("\n=== Complete Flow ===")


def connect_and_query(connection_string: str, query: str) -> list | None:
    """
    Demonstrate complete try/except/else/finally flow.
    """
    connection = None

    try:
        # Attempt to connect
        print(f"Connecting to {connection_string}...")
        if "invalid" in connection_string:
            raise ConnectionError("Invalid connection string")
        connection = {"active": True}  # Simulated connection

        # Execute query
        print(f"Executing: {query}")
        if "bad" in query.lower():
            raise ValueError("Invalid query")
        results = [{"id": 1}, {"id": 2}]  # Simulated results

    except ConnectionError as e:
        print(f"Connection failed: {e}")
        return None

    except ValueError as e:
        print(f"Query failed: {e}")
        return None

    else:
        # Only runs if no exception
        print(f"Query returned {len(results)} results")
        return results

    finally:
        # Always runs - cleanup
        if connection:
            print("Closing connection...")
            connection = None
        print("Done\n")


# Test scenarios
connect_and_query("postgresql://localhost", "SELECT * FROM users")
connect_and_query("invalid://connection", "SELECT * FROM users")
connect_and_query("postgresql://localhost", "BAD QUERY")


=== Complete Flow ===
Connecting to postgresql://localhost...
Executing: SELECT * FROM users
Query returned 2 results
Closing connection...
Done

Connecting to invalid://connection...
Connection failed: Invalid connection string
Done

Connecting to postgresql://localhost...
Executing: BAD QUERY
Query failed: Invalid query
Closing connection...
Done



## Exception Hierarchy


In [11]:
print("=== Exception Hierarchy ===")

print("""
Python Exception Hierarchy:

BaseException
├── SystemExit
├── KeyboardInterrupt
├── GeneratorExit
└── Exception                    ← Most exceptions inherit from here
    ├── StopIteration
    ├── ArithmeticError
    │   ├── ZeroDivisionError
    │   └── OverflowError
    ├── LookupError
    │   ├── IndexError
    │   └── KeyError
    ├── ValueError
    ├── TypeError
    ├── AttributeError
    ├── OSError (IOError)
    │   ├── FileNotFoundError
    │   └── PermissionError
    └── RuntimeError
        └── NotImplementedError
""")

# Catching parent catches all children
try:
    data = {"a": 1}
    value = data["b"]  # KeyError
except LookupError as e:  # Catches KeyError and IndexError
    print(f"LookupError caught: {type(e).__name__}: {e}")

=== Exception Hierarchy ===

Python Exception Hierarchy:

BaseException
├── SystemExit
├── KeyboardInterrupt
├── GeneratorExit
└── Exception                    ← Most exceptions inherit from here
    ├── StopIteration
    ├── ArithmeticError
    │   ├── ZeroDivisionError
    │   └── OverflowError
    ├── LookupError
    │   ├── IndexError
    │   └── KeyError
    ├── ValueError
    ├── TypeError
    ├── AttributeError
    ├── OSError (IOError)
    │   ├── FileNotFoundError
    │   └── PermissionError
    └── RuntimeError
        └── NotImplementedError

LookupError caught: KeyError: 'b'


## Raising Exceptions


In [12]:
print("\n=== Raising Exceptions ===")


def validate_batch_size(size: int) -> None:
    """Validate batch size parameter."""
    if not isinstance(size, int):
        raise TypeError(f"Batch size must be int, got {type(size).__name__}")

    if size <= 0:
        raise ValueError(f"Batch size must be positive, got {size}")

    if size > 10000:
        raise ValueError(f"Batch size too large: {size} (max 10000)")


# Test validation
test_values = [100, -5, 50000, "invalid"]

for val in test_values:
    try:
        validate_batch_size(val)
        print(f"✓ {val} is valid")
    except (TypeError, ValueError) as e:
        print(f"✗ {val}: {e}")

# Re-raising exceptions
print("\n--- Re-raising exceptions ---")


def process_with_logging(data):
    """Process data and re-raise exception after logging."""
    try:
        return 100 / data
    except Exception as e:
        print(f"Error occurred: {e}")
        raise  # Re-raise the same exception


try:
    process_with_logging(0)
except ZeroDivisionError:
    print("Caught re-raised exception")


=== Raising Exceptions ===
✓ 100 is valid
✗ -5: Batch size must be positive, got -5
✗ 50000: Batch size too large: 50000 (max 10000)
✗ invalid: Batch size must be int, got str

--- Re-raising exceptions ---
Error occurred: division by zero
Caught re-raised exception


## Bare Except — Don'T Do This!


In [13]:
print("\n=== Avoid Bare Except ===")


=== Avoid Bare Except ===


BAD: Catches everything including KeyboardInterrupt!
try:
    something()
except:  # DON'T DO THIS!
    pass
GOOD: Catch Exception (or specific exceptions)

In [15]:
try:
    result = 10 / 0
except Exception as e:
    print(f"Caught: {type(e).__name__}: {e}")

# Even better: Catch specific exceptions
try:
    result = 10 / 0
except ZeroDivisionError:
    print("Handled division by zero")

Caught: ZeroDivisionError: division by zero
Handled division by zero


## Practical: Robust Data Loader


In [16]:
print("\n=== Practical: Robust Data Loader ===")


def load_records(file_path: str, required_fields: list[str]) -> list[dict]:
    """
    Load records from file with comprehensive error handling.
    """
    records = []
    errors = []

    try:
        with open(file_path, "r") as f:
            import json
            data = json.load(f)

    except FileNotFoundError:
        print(f"File not found: {file_path}")
        # Return demo data for this example
        data = [
            {"id": 1, "name": "Alice", "value": 100},
            {"id": "invalid", "name": "Bob", "value": 200},
            {"id": 3, "value": 300},  # Missing name
        ]

    except json.JSONDecodeError as e:
        print(f"Invalid JSON in {file_path}: {e}")
        return []

    # Process records
    for i, record in enumerate(data):
        try:
            # Validate required fields
            for field in required_fields:
                if field not in record:
                    raise KeyError(f"Missing required field: {field}")

            # Validate types
            if not isinstance(record.get("id"), int):
                raise TypeError(f"ID must be integer, got {type(record.get('id')).__name__}")

            records.append(record)

        except (KeyError, TypeError) as e:
            errors.append({"index": i, "error": str(e)})

    print(f"Loaded {len(records)} valid records, {len(errors)} errors")
    if errors:
        print("Errors:")
        for err in errors:
            print(f"  Row {err['index']}: {err['error']}")

    return records


# Test the loader
records = load_records("data.json", ["id", "name"])
print(f"Valid records: {records}")


=== Practical: Robust Data Loader ===
File not found: data.json
Loaded 1 valid records, 2 errors
Errors:
  Row 1: ID must be integer, got str
  Row 2: 'Missing required field: name'
Valid records: [{'id': 1, 'name': 'Alice', 'value': 100}]


## Summary


In [17]:
print("\n=== Summary ===")
print("""
Basic Structure:
  try:
      # Code that might raise exception
  except ExceptionType as e:
      # Handle exception
  else:
      # Runs if no exception
  finally:
      # Always runs (cleanup)

Best Practices:
  1. Catch specific exceptions, not bare except
  2. Use else for success-only code
  3. Use finally for cleanup
  4. Re-raise after logging if needed
  5. Create custom exceptions for domain errors

Common Exceptions:
  ValueError:    Invalid value
  TypeError:     Wrong type
  KeyError:      Missing dict key
  IndexError:    Index out of range
  FileNotFoundError: File doesn't exist
  ZeroDivisionError: Division by zero
""")


=== Summary ===

Basic Structure:
  try:
      # Code that might raise exception
  except ExceptionType as e:
      # Handle exception
  else:
      # Runs if no exception
  finally:
      # Always runs (cleanup)

Best Practices:
  1. Catch specific exceptions, not bare except
  2. Use else for success-only code
  3. Use finally for cleanup
  4. Re-raise after logging if needed
  5. Create custom exceptions for domain errors

Common Exceptions:
  ValueError:    Invalid value
  TypeError:     Wrong type
  KeyError:      Missing dict key
  IndexError:    Index out of range
  FileNotFoundError: File doesn't exist
  ZeroDivisionError: Division by zero

